# LeetCode 2233 — Maximum Product After K Increments bằng Python

Link đề: https://leetcode.com/problems/maximum-product-after-k-increments/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def maximumProduct(self, nums: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra chiến lược tham lam **"luôn tăng phần tử nhỏ nhất"** và **chứng minh** nó bằng lập luận trao đổi (exchange argument).
2. Cài đặt bằng **min-heap** (`heapq.heapreplace`) — `O(k log n)`.
3. Tối ưu thành thuật toán **"đổ nước" (leveling)** sau khi sắp xếp — `O(n log n)`, **không phụ thuộc `k`**, kèm `pow(x, e, MOD)`.
4. Biết **khi nào lấy modulo**: so sánh/tối ưu trên giá trị thật, chỉ lấy mod khi tính tích cuối.
5. Kiểm thử bằng brute force liệt kê mọi cách phân phối `k` lần tăng (so trên tích **thật**, không mod).

## 2. Đề bài & ví dụ

Cho mảng số nguyên **không âm** `nums` và số nguyên `k`. Mỗi thao tác chọn **một** phần tử bất kỳ và tăng nó thêm `1`. Thực hiện **tối đa** `k` thao tác để **tích** các phần tử lớn nhất; trả về tích đó **modulo `10^9 + 7`** (tối đa hóa trước, rồi mới lấy mod).

**Ví dụ 1:** `nums = [0, 4], k = 5` → `20` (tăng phần tử đầu 5 lần: `[5, 4]`).

**Ví dụ 2:** `nums = [6, 3, 3, 2], k = 2` → `216` (`[6, 4, 3, 3]`).

**Ràng buộc:** `1 ≤ n, k ≤ 10^5`, `0 ≤ nums[i] ≤ 10^6`.

In [1]:
import heapq
import random
import time
from bisect import bisect_right
from functools import reduce
from itertools import product
from math import prod
from typing import List

MOD = 10**9 + 7
examples = [([0, 4], 5, 20), ([6, 3, 3, 2], 2, 216)]
for nums, k, out in examples:
    print(f"nums={nums}, k={k} -> {out}")

nums=[0, 4], k=5 -> 20
nums=[6, 3, 3, 2], k=2 -> 216


## 3. Kiến thức nền

### 3.1 Min-heap trong Python

`heapq` là **min-heap** trên list: `heapify` `O(n)`, `nums[0]` là nhỏ nhất, `heapreplace(h, x)` = pop nhỏ nhất rồi push `x` trong **một** lần sift (`O(log n)`) — nhanh hơn `heappop` + `heappush`.

In [2]:
h = [6, 3, 3, 2]
heapq.heapify(h)
print("heap:", h, "| nhỏ nhất:", h[0])
old = heapq.heapreplace(h, h[0] + 1)
print("heapreplace(h, 2+1) trả", old, "-> heap:", h)

heap: [2, 3, 3, 6] | nhỏ nhất: 2
heapreplace(h, 2+1) trả 2 -> heap: [3, 3, 3, 6]


### 3.2 Modulo và phép nhân

`(a·b) mod m = ((a mod m)·(b mod m)) mod m` ⇒ có thể lấy mod sau **mỗi** phép nhân để số không phình to. `pow(x, e, m)` tính lũy thừa modulo trong `O(log e)`.

**Nhưng:** mod **không bảo toàn thứ tự** — `a > b` không suy ra `a mod m > b mod m`. Vì thế mọi quyết định "cách nào tốt hơn" phải dựa trên **giá trị thật** (hoặc lập luận), không dựa trên tích đã mod.

In [3]:
a, b = MOD + 1, 5
print("a > b:", a > b, "| a % MOD > b % MOD:", a % MOD > b % MOD, f"({a % MOD} vs {b % MOD})")
print("pow(3, 10**18, MOD) =", pow(3, 10**18, MOD))

a > b: True | a % MOD > b % MOD: False (1 vs 5)
pow(3, 10**18, MOD) = 246336683


### 3.3 Tăng nhỏ nhất có lợi nhất — trực giác qua tỷ lệ

Tăng `x` lên `x + 1` nhân tích với `(x + 1)/x` (khi `x > 0`), tỷ lệ này **giảm** khi `x` tăng: `1/1 → 2`, `2 → 1.5`, `3 → 1.33`… Còn `x = 0` thì tích đang là 0, tăng nó là bắt buộc để tích khác 0.

In [4]:
for x in range(1, 7):
    print(f"x={x}: tăng lên {x + 1} nhân tích với {(x + 1) / x:.3f}")

x=1: tăng lên 2 nhân tích với 2.000
x=2: tăng lên 3 nhân tích với 1.500
x=3: tăng lên 4 nhân tích với 1.333
x=4: tăng lên 5 nhân tích với 1.250
x=5: tăng lên 6 nhân tích với 1.200
x=6: tăng lên 7 nhân tích với 1.167


## 4. Ý tưởng nền tảng / trực giác

Xét hai phần tử `x ≤ y`, có **một** lần tăng: tăng `x` cho `(x+1)·y = xy + y`, tăng `y` cho `x·(y+1) = xy + x`. Vì `y ≥ x` ⇒ tăng **phần tử nhỏ hơn** luôn không tệ hơn. Các phần tử còn lại chỉ là hệ số nhân chung (không âm) ⇒ kết luận giữ nguyên cho cả mảng.

Hệ quả: sau khi dùng hết `k` lần, các phần tử "được tăng" sẽ **san bằng** nhau (chênh nhau tối đa 1) — giống đổ nước vào các cột thấp nhất.

| Bước | Mảng (ví dụ 2) | Tăng | Tích |
|---|---|---|---|
| 0 | [6, 3, 3, 2] | — | 108 |
| 1 | [6, 3, 3, **3**] | 2 → 3 | 162 |
| 2 | [6, **4**, 3, 3] | 3 → 4 | 216 |

In [5]:
arr = [6, 3, 3, 2]
print("bắt đầu:", arr, "tích", prod(arr))
for step in range(1, 3):
    i = arr.index(min(arr)); arr[i] += 1
    print(f"bước {step}: tăng vị trí {i} -> {arr}, tích {prod(arr)}")

bắt đầu: [6, 3, 3, 2] tích 108
bước 1: tăng vị trí 3 -> [6, 3, 3, 3], tích 162
bước 2: tăng vị trí 1 -> [6, 4, 3, 3], tích 216


### 4.1 Vì sao luôn dùng **hết** `k` lần?

Đề nói "tối đa `k`". Nhưng các số không âm: tăng một phần tử không bao giờ làm tích **giảm** (`(x+1)·P ≥ x·P` với `P ≥ 0`). Vậy dùng đúng `k` lần luôn tối ưu (có thể bằng nhau khi tích có thừa số 0 khác — nhưng tham lam tăng 0 trước nên không xảy ra mất mát).

## 5. Cách 1 — Brute Force / baseline: liệt kê mọi phân phối

**Ý tưởng:** liệt kê mọi bộ `(c_0, …, c_{n−1})` với `c_i ≥ 0`, `Σ c_i ≤ k`, tính tích **thật** `∏ (nums[i] + c_i)`, lấy max, cuối cùng mới mod. Chỉ dùng cho `n, k` nhỏ.

In [6]:
def max_product_brute(nums, k):
    n = len(nums); best = -1
    for cs in product(range(k + 1), repeat=n):
        if sum(cs) <= k:
            best = max(best, prod(x + c for x, c in zip(nums, cs)))
    return best % MOD

for nums, k, out in examples:
    got = max_product_brute(nums, k); print(nums, k, "->", got); assert got == out

[0, 4] 5 -> 20
[6, 3, 3, 2] 2 -> 216


**Độ phức tạp:** `(k+1)^n` bộ × `O(n)` ⇒ bùng nổ tổ hợp, chỉ dùng kiểm thử (`n ≤ 4`, `k ≤ 6`).

**Nút thắt:** thử mọi phân phối trong khi lập luận ở mục 4 cho thấy mỗi lần tăng chỉ có **một** lựa chọn tốt nhất: phần tử nhỏ nhất.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tham lam):** mỗi lần tăng, chọn phần tử **nhỏ nhất** hiện tại. Cần lấy-nhỏ-nhất-và-cập-nhật lặp lại `k` lần ⇒ **min-heap**.

**Insight 2 (đổ nước):** kết quả tham lam có cấu trúc rõ ràng: sắp xếp tăng dần `a_0 ≤ a_1 ≤ …`. Sẽ có một **mức** `L` và số `c` sao cho `c` phần tử nhỏ nhất bị nâng lên `L` (hoặc `L+1`), phần còn lại giữ nguyên (đều `≥ L`). Chi phí nâng `c` phần tử đầu lên mức `L` là `c·L − (a_0 + … + a_{c−1})`. Chọn `L` lớn nhất mà chi phí `≤ k`; phần dư `r` (< `c`) cộng thêm 1 cho `r` phần tử trong nhóm.

**Quy tắc "nếu… thì…":**
- **Nếu** cần "k lần lấy min" và `k` vừa phải **thì** heap.
- **Nếu** `k` rất lớn (vd `10^18`) **thì** đổ nước: `O(n log n)` + `pow` modulo.
- **Nếu** đề có mod **thì** quyết định trên giá trị thật/lập luận, mod chỉ ở bước tính tích.

### 6.1 Hai cách nhìn cùng một kết quả

Heap mô phỏng **từng giọt nước**; đổ nước tính thẳng **mực nước cuối**. Ví dụ `nums = [0, 4], k = 5`: heap tăng `0 → 1 → 2 → 3 → 4` (4 lần), lúc này hai phần tử bằng nhau `[4, 4]`, lần thứ 5 tăng một trong hai ⇒ `[5, 4]`. Đổ nước: nhóm `[0, 4]` (vì nâng `0` lên `4` tốn 4 ≤ 5), `total = 5 + 4 = 9`, `L, r = divmod(9, 2) = (4, 1)` ⇒ một số `4`, một số `5` — trùng khớp, chỉ khác là không cần lặp `k` lần.

Điều này quan trọng khi `k` lớn: heap tốn `O(k log n)`, đổ nước không phụ thuộc `k`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Min-heap (bản nộp chính)

In [7]:
def max_product_heap(nums, k):
    h = list(nums)
    heapq.heapify(h)
    for _ in range(k):
        heapq.heapreplace(h, h[0] + 1)          # tăng phần tử nhỏ nhất
    ans = 1
    for x in h:
        ans = ans * x % MOD
    return ans

for nums, k, out in examples:
    assert max_product_heap(nums, k) == out
print([max_product_heap(n, k) for n, k, _ in examples])

[20, 216]


**Độ phức tạp Cách 2:** `heapify O(n)` + `k` lần `heapreplace O(log n)` + tích `O(n)` ⇒ `O(n + k log n)`. Space `O(n)`.

### 7.2 Cách 3 — Sắp xếp + đổ nước (tính trực tiếp mức cuối)

Duyệt `c = 1, 2, …`: giữ `pre = a_0 + … + a_{c−1}`. Mức tối đa đạt được bằng `c` phần tử đầu là `L = (k + pre) // c`, nhưng chỉ hợp lệ nếu `L ≤ a_c` (không vượt phần tử kế tiếp — nếu vượt thì nên gộp thêm phần tử đó vào nhóm). Chọn `c` lớn nhất thỏa mãn điều kiện "nâng đủ tới `a_{c-1}`".

In [8]:
def max_product_level(nums, k):
    a = sorted(nums); n = len(a)
    pre = 0; c = 0
    # gộp dần: nhóm c phần tử đầu có thể được nâng lên tới a[c] hay không?
    while c < n:
        pre += a[c]; c += 1
        if c == n or a[c] * c - pre > k:      # không đủ để nâng cả nhóm lên a[c] -> dừng
            break
    total = k + pre                           # tổng của nhóm sau khi dùng hết k
    L, r = divmod(total, c)                   # r phần tử mức L+1, c-r phần tử mức L
    ans = pow(L, c - r, MOD) * pow(L + 1, r, MOD) % MOD
    for x in a[c:]:
        ans = ans * x % MOD
    return ans

for nums, k, out in examples:
    assert max_product_level(nums, k) == out
print([max_product_level(n, k) for n, k, _ in examples])

[20, 216]


**Vì sao `L ≤ a[c]` tự động?** Vòng lặp dừng khi không đủ để nâng cả nhóm lên `a[c]`, tức `c·a[c] − pre > k` ⇒ `k + pre < c·a[c]` ⇒ `L = (k+pre)//c < a[c]`, còn `L + 1 ≤ a[c]`: các phần tử ngoài nhóm không nhỏ hơn mức mới — đúng cấu trúc tham lam. Đồng thời ở bước trước đã đủ để nâng lên `a[c−1]` ⇒ `L ≥ a[c−1]`: không phần tử nào trong nhóm bị "hạ".

**Độ phức tạp Cách 3:** `O(n log n)` (sắp xếp) + `O(n)` + `O(log)` cho `pow` ⇒ **không phụ thuộc `k`**. Space `O(n)`.

### 7.3 Cách 4 — Tìm nhị phân trên mức `L`

Hàm `cost(L) = Σ max(0, L − a_i)` tăng theo `L`. Tìm `L` lớn nhất với `cost(L) ≤ k`, rồi phân phối phần dư cho các phần tử **đang ở đúng mức `L`** (đều là phần tử đã được nâng hoặc vốn bằng `L`).

In [9]:
def max_product_bs(nums, k):
    a = sorted(nums)
    cost = lambda L: sum(L - x for x in a if x < L)
    lo, hi = a[0], a[0] + k                   # cost(a[0]) = 0 <= k
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if cost(mid) <= k: lo = mid
        else: hi = mid - 1
    L = lo
    b = [max(x, L) for x in a]
    r = k - cost(L)                           # r < số phần tử có b_i == L (nếu không thì L+1 cũng khả thi)
    cnt_L = b.count(L)
    assert r < cnt_L
    ans = pow(L + 1, r, MOD) * pow(L, cnt_L - r, MOD) % MOD
    for x in b:
        if x != L: ans = ans * x % MOD
    return ans

for nums, k, out in examples:
    assert max_product_bs(nums, k) == out
print([max_product_bs(n, k) for n, k, _ in examples])

[20, 216]


**Độ phức tạp Cách 4:** `O(n log n + n log k)` (mỗi lần tính `cost` là `O(n)`; có thể `O(log n)` với prefix sum + `bisect`). Space `O(n)`.

**Vì sao `r < cnt_L`?** Nếu `r ≥ cnt_L` thì nâng mọi phần tử đang ở mức `L` thêm 1 tốn `cnt_L ≤ r` ⇒ `cost(L+1) ≤ k`, mâu thuẫn với `L` lớn nhất (phần tử `> L` không tốn gì khi lên `L+1`).

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 `nums = [6, 3, 3, 2], k = 2`:

| Lần | heap trước | min | heap sau |
|---|---|---|---|
| 1 | [2, 3, 3, 6] | 2 | [3, 3, 3, 6] |
| 2 | [3, 3, 3, 6] | 3 | [3, 3, 4, 6] (thứ tự trong list có thể khác) |

Tích `= 3·3·4·6 = 216`.

In [10]:
def trace_heap(nums, k):
    h = list(nums); heapq.heapify(h)
    print("heapify:", h)
    for step in range(1, k + 1):
        m = h[0]
        heapq.heapreplace(h, m + 1)
        print(f"lần {step}: min = {m} -> {m + 1} | heap (list) = {h} | đã sắp: {sorted(h)}")
    ans = 1
    for x in h:
        ans = ans * x % MOD
    print("tích =", " × ".join(map(str, sorted(h))), "=", ans)
    return ans

trace_heap([6, 3, 3, 2], 2)

heapify: [2, 3, 3, 6]
lần 1: min = 2 -> 3 | heap (list) = [3, 3, 3, 6] | đã sắp: [3, 3, 3, 6]
lần 2: min = 3 -> 4 | heap (list) = [3, 3, 4, 6] | đã sắp: [3, 3, 4, 6]
tích = 3 × 3 × 4 × 6 = 216


216

In [11]:
trace_heap([0, 4], 5)

heapify: [0, 4]
lần 1: min = 0 -> 1 | heap (list) = [1, 4] | đã sắp: [1, 4]
lần 2: min = 1 -> 2 | heap (list) = [2, 4] | đã sắp: [2, 4]
lần 3: min = 2 -> 3 | heap (list) = [3, 4] | đã sắp: [3, 4]
lần 4: min = 3 -> 4 | heap (list) = [4, 4] | đã sắp: [4, 4]
lần 5: min = 4 -> 5 | heap (list) = [4, 5] | đã sắp: [4, 5]
tích = 4 × 5 = 20


20

Trace Cách 3 trên `nums = [1, 2, 5, 5, 9], k = 7` (tự dựng — đã kiểm bằng brute force bên dưới):

| c | nhóm | pre | cần để nâng nhóm lên `a[c]` | ≤ k? |
|---|---|---|---|---|
| 1 | [1] | 1 | 1·2 − 1 = 1 | có |
| 2 | [1, 2] | 3 | 2·5 − 3 = 7 | có |
| 3 | [1, 2, 5] | 8 | 3·5 − 8 = 7 | có |
| 4 | [1, 2, 5, 5] | 13 | 4·9 − 13 = 23 | **không** → dừng |

`total = 7 + 13 = 20`, `L, r = divmod(20, 4) = (5, 0)` ⇒ nhóm thành `[5, 5, 5, 5]`, tích `5^4 · 9 = 5625`.

In [12]:
def trace_level(nums, k):
    a = sorted(nums); n = len(a); pre = 0; c = 0
    print("sắp xếp:", a)
    while c < n:
        pre += a[c]; c += 1
        need = a[c] * c - pre if c < n else None
        print(f"c={c}: nhóm={a[:c]}, pre={pre}, cần để lên a[c]={a[c] if c < n else '-'}: {need}",
              "-> dừng" if c == n or need > k else "-> gộp tiếp")
        if c == n or need > k: break
    L, r = divmod(k + pre, c)
    print(f"total={k + pre}, L={L}, r={r}: {c - r} phần tử = {L}, {r} phần tử = {L + 1}, giữ nguyên {a[c:]}")
    ans = pow(L, c - r, MOD) * pow(L + 1, r, MOD) % MOD
    for x in a[c:]: ans = ans * x % MOD
    print("tích =", ans)
    return ans

assert trace_level([1, 2, 5, 5, 9], 7) == max_product_brute([1, 2, 5, 5, 9], 7) == 5625

sắp xếp: [1, 2, 5, 5, 9]
c=1: nhóm=[1], pre=1, cần để lên a[c]=2: 1 -> gộp tiếp
c=2: nhóm=[1, 2], pre=3, cần để lên a[c]=5: 7 -> gộp tiếp
c=3: nhóm=[1, 2, 5], pre=8, cần để lên a[c]=5: 7 -> gộp tiếp
c=4: nhóm=[1, 2, 5, 5], pre=13, cần để lên a[c]=9: 23 -> dừng
total=20, L=5, r=0: 4 phần tử = 5, 0 phần tử = 6, giữ nguyên [9]
tích = 5625


In [13]:
assert trace_level([6, 3, 3, 2], 2) == 216

sắp xếp: [2, 3, 3, 6]
c=1: nhóm=[2], pre=2, cần để lên a[c]=3: 1 -> gộp tiếp
c=2: nhóm=[2, 3], pre=5, cần để lên a[c]=3: 1 -> gộp tiếp
c=3: nhóm=[2, 3, 3], pre=8, cần để lên a[c]=6: 10 -> dừng
total=10, L=3, r=1: 2 phần tử = 3, 1 phần tử = 4, giữ nguyên [6]
tích = 216


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maximumProduct(self, nums: List[int], k: int) -> int:
        MOD = 10**9 + 7
        heapq.heapify(nums)                      # min-heap tại chỗ
        for _ in range(k):
            heapq.heapreplace(nums, nums[0] + 1) # tăng phần tử nhỏ nhất
        ans = 1
        for x in nums:
            ans = ans * x % MOD                  # mod chỉ khi tính tích
        return ans

for nums, k, out in examples:
    assert Solution().maximumProduct(list(nums), k) == out
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề trao đổi.** Xét một phân phối tối ưu `c = (c_0, …, c_{n−1})` (`Σ c_i = k`), mảng cuối `b_i = a_i + c_i`. Giả sử có `i, j` với `c_j > 0` và `b_i < b_j − 1`, tức phần tử `j` đã được tăng nhưng phần tử `i` cuối cùng nhỏ hơn `b_j` ít nhất 2. Chuyển một lần tăng từ `j` sang `i`: `(b_i + 1)(b_j − 1) = b_i b_j + b_j − b_i − 1 > b_i b_j` (vì `b_j − b_i ≥ 2`) — nhân với tích các phần tử khác (≥ 0) thì không giảm. Vậy luôn có một nghiệm tối ưu thỏa: **mọi phần tử được tăng đều có giá trị cuối ≤ min(b) + 1**, tức mảng cuối là kết quả "đổ nước".

**Tham lam heap tạo đúng cấu trúc đó.** Mỗi bước tăng phần tử nhỏ nhất ⇒ phần tử được tăng có giá trị cuối không vượt quá (min hiện tại + 1) ⇒ chênh lệch trong nhóm được tăng ≤ 1, và nhóm này luôn là các phần tử nhỏ nhất. Cấu trúc "đổ nước" với tổng cố định là **duy nhất** (theo multiset), nên tham lam cho cùng tích với nghiệm tối ưu.

**Chi tiết số 0.** Với `b_i = 0`, công thức trao đổi vẫn đúng (`(0+1)(b_j−1) ≥ 0 = 0·b_j`). Tham lam tăng 0 trước nên tích khác 0 ngay khi `k ≥` số lượng số 0.

**Modulo.** Thuật toán không so sánh tích nào; nó dựng ra mảng tối ưu rồi mới tính tích mod ⇒ "tối đa hóa trước, rồi lấy mod" đúng như đề.

### 10.1 Kiểm chứng bằng số: mảng cuối tối ưu luôn "đều" (chênh ≤ 1 trong nhóm được tăng)

Ta liệt kê **mọi** phân phối tối ưu (theo brute force) trên dữ liệu nhỏ và kiểm: (a) mọi phần tử được tăng có giá trị cuối `≤ min(b) + 1`; (b) multiset mảng cuối trùng với kết quả của heap — trừ trường hợp tích bằng 0 (khi đó nhiều mảng khác nhau cùng tích 0).

In [15]:
def optimal_finals(nums, k):
    best = -1; finals = []
    for cs in product(range(k + 1), repeat=len(nums)):
        if sum(cs) == k:
            b = [x + c for x, c in zip(nums, cs)]; v = prod(b)
            if v > best: best, finals = v, [(cs, b)]
            elif v == best: finals.append((cs, b))
    return best, finals

rngc = random.Random(77); checked = 0
for _ in range(200):
    nums = [rngc.randint(0, 5) for _ in range(rngc.randint(1, 3))]; k = rngc.randint(1, 5)
    best, finals = optimal_finals(nums, k)
    if best == 0: continue
    for cs, b in finals:
        assert all(bi <= min(b) + 1 for bi, c in zip(b, cs) if c > 0)
        h = list(nums); heapq.heapify(h)
        for _ in range(k): heapq.heapreplace(h, h[0] + 1)
        assert sorted(b) == sorted(h)
    checked += 1
print(f"Đã kiểm {checked} bộ dữ liệu: mọi nghiệm tối ưu đều có dạng đổ nước")

Đã kiểm 196 bộ dữ liệu: mọi nghiệm tối ưu đều có dạng đổ nước


## 11. Phân tích Time / Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O((k+1)^n · n)` | `O(n)` |
| 2. Min-heap | `O(n + k log n)` | `O(n)` (tại chỗ nếu sửa `nums`) |
| 3. Sắp xếp + đổ nước | `O(n log n)`, không phụ thuộc `k` | `O(n)` |
| 4. Tìm nhị phân mức | `O(n log n + n log k)` | `O(n)` |

Với `n = k = 10^5`: heap ~ `1.7·10^6` thao tác — đủ nhanh. Đo thực tế:

In [16]:
rng = random.Random(2233)
big = [rng.randint(0, 10**6) for _ in range(10**5)]
for name, fn in [("heap", max_product_heap), ("level", max_product_level), ("binary search", max_product_bs)]:
    t = time.perf_counter(); v = fn(big, 10**5); dt = time.perf_counter() - t
    print(f"{name:13s}: {v} trong {dt:.3f}s")
t = time.perf_counter(); v = max_product_level(big, 10**15); print(f"level với k = 10^15: {v} trong {time.perf_counter() - t:.3f}s (heap không làm nổi)")

heap         : 441424676 trong 0.036s
level        : 441424676 trong 0.024s


binary search: 441424676 trong 0.085s


level với k = 10^15: 852710479 trong 0.031s (heap không làm nổi)


## 12. Bộ test

So mọi cách với brute force (tích thật, mod cuối) trên dữ liệu nhỏ ngẫu nhiên seed cố định, cộng ví dụ đề và biên: toàn số 0, một phần tử, `k` rất lớn, phần tử bằng nhau.

In [17]:
fns = {"heap": max_product_heap, "level": max_product_level, "bs": max_product_bs,
       "Solution": lambda a, k: Solution().maximumProduct(list(a), k)}
edge = [([0], 1, 1), ([0, 0, 0], 2, 0), ([0, 0, 0], 3, 1), ([5], 10, 15), ([1, 1, 1], 4, 12), ([10**6], 1, (10**6 + 1) % MOD),
        ([2, 2], 1, 6), ([0, 10], 3, 30)]
for nums, k, out in examples + edge:
    assert max_product_brute(nums, k) == out, (nums, k)
    for name, fn in fns.items():
        assert fn(list(nums), k) == out, (name, nums, k)
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [18]:
rng = random.Random(42)
for _ in range(400):
    n = rng.randint(1, 4); k = rng.randint(1, 6)
    nums = [rng.randint(0, 6) for _ in range(n)]
    exp = max_product_brute(nums, k)
    for name, fn in fns.items():
        assert fn(list(nums), k) == exp, (name, nums, k)
print("400 test nhỏ ngẫu nhiên: mọi cách khớp brute force")

400 test nhỏ ngẫu nhiên: mọi cách khớp brute force


In [19]:
for _ in range(60):
    n = rng.randint(1, 300); k = rng.randint(1, 3000)
    nums = [rng.randint(0, 10**6) if rng.random() < 0.5 else rng.randint(0, 20) for _ in range(n)]
    vals = {name: fn(list(nums), k) for name, fn in fns.items()}
    assert len(set(vals.values())) == 1, vals
print("60 test vừa: heap, level, binary search, Solution trùng nhau")

60 test vừa: heap, level, binary search, Solution trùng nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tăng phần tử **lớn nhất** (hoặc cứ tăng phần tử đầu)

In [20]:
def wrong_increase_max(nums, k):
    a = list(nums)
    for _ in range(k):
        a[a.index(max(a))] += 1                                   # SAI: nên tăng min
    return prod(a) % MOD

bad = next((nums, k) for n in range(1, 4) for nums in product(range(4), repeat=n) for k in range(1, 4)
           if wrong_increase_max(nums, k) != max_product_brute(nums, k))
print("phản ví dụ:", bad, "sai =", wrong_increase_max(*bad), "| đúng =", max_product_brute(*bad))

phản ví dụ: ((0, 0), 2) sai = 0 | đúng = 1


In [21]:
print("sửa (tăng min):", max_product_heap(list(bad[0]), bad[1]))

sửa (tăng min): 1


### Lỗi 2 — Dùng tích **đã mod** để so sánh/chọn

Bản sai thử "tăng phần tử nào cho tích mod lớn nhất" ở mỗi bước. Khi tích vượt `MOD`, so sánh trên mod là vô nghĩa. Ta tìm phản ví dụ với các số lớn.

In [22]:
def wrong_mod_compare(nums, k):
    a = list(nums)
    for _ in range(k):
        best_i = max(range(len(a)), key=lambda i: prod(a[:i] + [a[i] + 1] + a[i + 1:]) % MOD)   # SAI
        a[best_i] += 1
    return prod(a) % MOD

rngw = random.Random(5)
found = None
for _ in range(3000):
    nums = [rngw.randint(10**5, 10**6) for _ in range(rngw.randint(2, 3))]; k = rngw.randint(1, 3)
    if wrong_mod_compare(nums, k) != max_product_heap(nums, k):
        found = (nums, k); break
print("phản ví dụ:", found, "| sai =", wrong_mod_compare(*found), "| đúng =", max_product_heap(*found))

phản ví dụ: ([877820, 475951, 933820], 3) | sai = 361518667 | đúng = 141206965


In [23]:
print("sửa (quyết định không dựa trên mod):", max_product_level(*found))

sửa (quyết định không dựa trên mod): 141206965


### Lỗi 3 — `heappop` rồi quên `heappush` (hoặc tăng `nums[0]` mà không sift)

Tăng `h[0] += 1` trực tiếp phá **bất biến heap**: phần tử đầu không còn chắc là nhỏ nhất ⇒ lần sau tăng sai phần tử.

In [24]:
def wrong_no_sift(nums, k):
    h = list(nums); heapq.heapify(h)
    for _ in range(k):
        h[0] += 1                                                 # SAI: không khôi phục heap
    return prod(h) % MOD

bad = next((nums, k) for n in range(2, 4) for nums in product(range(4), repeat=n) for k in range(1, 4)
           if wrong_no_sift(nums, k) != max_product_brute(nums, k))
print("phản ví dụ:", bad, "sai =", wrong_no_sift(*bad), "| đúng =", max_product_brute(*bad))

phản ví dụ: ((0, 0), 2) sai = 0 | đúng = 1


In [25]:
print("sửa (heapreplace):", max_product_heap(list(bad[0]), bad[1]))

sửa (heapreplace): 1


### Lỗi 4 — Đổ nước nhưng quên phân phối phần dư `r`

Lấy `L = (k + pre) // c` rồi gán cả nhóm bằng `L`, bỏ mất `r = (k + pre) % c` lần tăng.

In [26]:
def wrong_level_no_remainder(nums, k):
    a = sorted(nums); n = len(a); pre = 0; c = 0
    while c < n:
        pre += a[c]; c += 1
        if c == n or a[c] * c - pre > k: break
    L = (k + pre) // c
    ans = pow(L, c, MOD)                                          # SAI: thiếu (L+1)^r
    for x in a[c:]: ans = ans * x % MOD
    return ans

bad = next((nums, k) for n in range(1, 4) for nums in product(range(4), repeat=n) for k in range(1, 4)
           if wrong_level_no_remainder(nums, k) != max_product_brute(nums, k))
print("phản ví dụ:", bad, "sai =", wrong_level_no_remainder(*bad), "| đúng =", max_product_brute(*bad))

phản ví dụ: ((0, 0), 3) sai = 1 | đúng = 2


In [27]:
print("sửa:", max_product_level(list(bad[0]), bad[1]))

sửa: 2


### Lỗi 5 — Dừng gộp nhóm quá sớm (so với `a[c-1]` thay vì `a[c]`)

Điều kiện dừng phải hỏi "có đủ để nâng cả nhóm lên **phần tử kế tiếp** `a[c]` không". Bản sai kiểm điều kiện với phần tử cuối của nhóm (luôn đủ) theo kiểu khác và gộp **thiếu**/thừa.

In [28]:
def wrong_level_early(nums, k):
    a = sorted(nums); n = len(a); pre = 0; c = 0
    while c < n:
        pre += a[c]; c += 1
        if c == n or a[c] * c - pre >= k: break                   # SAI: '>=' dừng cả khi vừa đủ
    L, r = divmod(k + pre, c)
    ans = pow(L, c - r, MOD) * pow(L + 1, r, MOD) % MOD
    for x in a[c:]: ans = ans * x % MOD
    return ans

bad = next(((nums, k) for n in range(2, 4) for nums in product(range(5), repeat=n) for k in range(1, 6)
            if wrong_level_early(nums, k) != max_product_brute(nums, k)), None)
print("phản ví dụ:", bad, "(None = không tìm thấy trong mọi mảng n ≤ 3, giá trị ≤ 4, k ≤ 5)")

phản ví dụ: None (None = không tìm thấy trong mọi mảng n ≤ 3, giá trị ≤ 4, k ≤ 5)


Kết quả trên là `None` — tìm kiếm vét cạn không thấy phản ví dụ nào: lỗi "`>=` thay `>`" thực ra **vô hại** — khi vừa đủ để nâng lên `a[c]`, dừng sớm cho nhóm toàn `a[c]` với `r = 0`, cũng chính là mảng đổ nước đúng. Đây là ví dụ tốt về việc **kiểm bằng brute force** trước khi kết luận một thay đổi là bug. Lỗi thật sự là so với **phần tử cuối nhóm** `a[c-1]`:

In [29]:
def wrong_level_prev(nums, k):
    a = sorted(nums); n = len(a); pre = 0; c = 0
    while c < n:
        pre += a[c]; c += 1
        if c == n or a[c - 1] * c - pre > k: break                # SAI: luôn <= k nên gộp hết mọi phần tử
    L, r = divmod(k + pre, c)
    ans = pow(L, c - r, MOD) * pow(L + 1, r, MOD) % MOD
    for x in a[c:]: ans = ans * x % MOD
    return ans

bad = next((nums, k) for n in range(2, 4) for nums in product(range(5), repeat=n) for k in range(1, 6)
           if wrong_level_prev(nums, k) != max_product_brute(nums, k))
print("phản ví dụ:", bad, "sai =", wrong_level_prev(*bad), "| đúng =", max_product_brute(*bad))
print("sửa:", max_product_level(list(bad[0]), bad[1]))

phản ví dụ: ((0, 3), 1) sai = 4 | đúng = 3
sửa: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa hóa **tích** với tổng tăng cố định" ⇒ **san bằng** (làm các số càng đều càng tốt) — cùng họ với bất đẳng thức AM–GM.
- "Lặp `k` lần: lấy min/max rồi cập nhật" ⇒ heap với `heapreplace`.
- `k` lớn tới `10^9`/`10^18` ⇒ đổi sang công thức đóng (đổ nước, tìm nhị phân mức) + `pow(x, e, MOD)`.
- Đề có "maximize before modulo" ⇒ đừng so sánh giá trị đã mod.
- Python: tích dùng vòng lặp `ans = ans * x % MOD` (hoặc `reduce`) thay vì `math.prod` rồi mới mod — tránh số khổng lồ (10^5 số cỡ 10^6 ⇒ ~600.000 chữ số).

## 15. Biến thể của bài

1. **Tối thiểu hóa tổng bình phương sau `k` lần giảm** (LeetCode 2333 — Minimum Sum of Squared Difference): đổ nước ngược — luôn giảm phần tử **lớn nhất**.
2. **LeetCode 1648 — Sell Diminishing-Valued Colored Balls:** lấy max `k` lần, `k` tới `10^9` ⇒ tính theo "tầng" + công thức cấp số cộng.
3. **LeetCode 1675 / 2530 — Maximal Score After Applying K Operations:** heap lặp `k` lần.
4. **Tăng có chi phí khác nhau:** chuyển sang quy hoạch động hoặc tham lam theo tỷ lệ lợi ích.
5. **Mỗi phần tử tăng tối đa `cap` lần** (bài tập 5).

Minh họa biến thể 1 (LeetCode 2333 rút gọn: chỉ một mảng `d`, `k` lần giảm 1, không xuống dưới 0, tối thiểu tổng bình phương):

In [30]:
def min_sum_squares(d, k):
    h = [-x for x in d]; heapq.heapify(h)
    for _ in range(k):
        if h[0] == 0: break
        heapq.heapreplace(h, h[0] + 1)          # giảm phần tử lớn nhất
    return sum(x * x for x in h)

def min_sum_squares_brute(d, k):
    best = None
    for cs in product(range(k + 1), repeat=len(d)):
        if sum(cs) <= k and all(c <= x for c, x in zip(cs, d)):
            v = sum((x - c) ** 2 for x, c in zip(d, cs)); best = v if best is None else min(best, v)
    return best

rngv = random.Random(3)
for _ in range(200):
    d = [rngv.randint(0, 5) for _ in range(rngv.randint(1, 3))]; k = rngv.randint(0, 5)
    assert min_sum_squares(d, k) == min_sum_squares_brute(d, k)
print("biến thể 1 khớp brute force; ví dụ [3, 1, 2], k=2 ->", min_sum_squares([3, 1, 2], 2))

biến thể 1 khớp brute force; ví dụ [3, 1, 2], k=2 -> 6


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi phân phối | mũ | `O(n)` | kiểm thử |
| 2. Min-heap | tăng min `k` lần | `O(n + k log n)` | `O(n)` | **nộp bài**, dễ viết |
| 3. Đổ nước | sắp xếp, tính mức cuối | `O(n log n)` | `O(n)` | `k` rất lớn |
| 4. Tìm nhị phân mức | `cost(L) ≤ k` | `O(n log n + n log k)` | `O(n)` | khi khó suy công thức gộp |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `maximumProduct([1, 1, 1], 3)`.
2. **Tính tay:** trace heap cho `nums = [0, 0, 5], k = 4` — mảng cuối và tích.
3. **Điền TODO:** `final_array(nums, k)` trả về mảng cuối (đã sắp xếp) theo đổ nước, không dùng heap.
4. **Tự viết lại:** viết lại Cách 2 không nhìn code.
5. **Mini challenge:** mỗi phần tử chỉ được tăng tối đa `cap` lần — `max_product_cap(nums, k, cap)`. Kiểm bằng brute force.

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def final_array(nums, k):
    # TODO
    pass

In [32]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_maximum_product(nums, k):
    # TODO
    pass

In [33]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def max_product_cap(nums, k, cap):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Mỗi phần tử tăng 1 lần. (2) `[2, 2, 2]`. (3) `8`.

**Bài 2.** (1) Hai số 0 được tăng luân phiên. (2) `[0,0,5] → [1,0,5] → [1,1,5] → [2,1,5] → [2,2,5]`. (3) Tích `20`.

**Bài 3.** (1) Dùng vòng gộp nhóm của Cách 3. (2) `L, r = divmod(k + pre, c)`. (3) Nhóm gồm `c − r` số `L` và `r` số `L+1`, nối với `a[c:]`.

**Bài 4.** (1) `heapify`. (2) `heapreplace(h, h[0] + 1)` `k` lần. (3) Tích mod.

**Bài 5.** (1) Lập luận trao đổi vẫn đúng giữa các phần tử còn "quota". (2) Heap các cặp `(giá trị, số lần đã tăng)`; phần tử hết quota thì bỏ khỏi heap. (3) Brute force: `product(range(cap+1), repeat=n)`.

## 19. Đáp án tham khảo

In [34]:
# Bài 1, 2
assert Solution().maximumProduct([1, 1, 1], 3) == 8
assert trace_heap([0, 0, 5], 4) == 20 == max_product_brute([0, 0, 5], 4)
print("Bài 1, 2: OK")

heapify: [0, 0, 5]
lần 1: min = 0 -> 1 | heap (list) = [0, 1, 5] | đã sắp: [0, 1, 5]
lần 2: min = 0 -> 1 | heap (list) = [1, 1, 5] | đã sắp: [1, 1, 5]
lần 3: min = 1 -> 2 | heap (list) = [1, 2, 5] | đã sắp: [1, 2, 5]
lần 4: min = 1 -> 2 | heap (list) = [2, 2, 5] | đã sắp: [2, 2, 5]
tích = 2 × 2 × 5 = 20
Bài 1, 2: OK


In [35]:
# Bài 3, 4
def final_array_ans(nums, k):
    a = sorted(nums); n = len(a); pre = 0; c = 0
    while c < n:
        pre += a[c]; c += 1
        if c == n or a[c] * c - pre > k: break
    L, r = divmod(k + pre, c)
    return sorted([L] * (c - r) + [L + 1] * r + a[c:])

def final_array_heap(nums, k):
    h = list(nums); heapq.heapify(h)
    for _ in range(k): heapq.heapreplace(h, h[0] + 1)
    return sorted(h)

def my_maximum_product_ans(nums, k):
    h = list(nums); heapq.heapify(h)
    for _ in range(k): heapq.heapreplace(h, h[0] + 1)
    return reduce(lambda x, y: x * y % MOD, h, 1)

rng5 = random.Random(9)
for _ in range(300):
    nums = [rng5.randint(0, 30) for _ in range(rng5.randint(1, 8))]; k = rng5.randint(1, 60)
    assert final_array_ans(nums, k) == final_array_heap(nums, k)
    assert my_maximum_product_ans(nums, k) == max_product_level(nums, k)
print("Bài 3, 4: OK; final_array([0, 0, 5], 4) =", final_array_ans([0, 0, 5], 4))

Bài 3, 4: OK; final_array([0, 0, 5], 4) = [2, 2, 5]


In [36]:
# Bài 5
def max_product_cap_ans(nums, k, cap):
    h = [(x, 0) for x in nums]; heapq.heapify(h); done = []
    while k and h:
        x, used = heapq.heappop(h)
        x += 1; used += 1; k -= 1
        if used < cap: heapq.heappush(h, (x, used))
        else: done.append(x)
    vals = [x for x, _ in h] + done
    return reduce(lambda p, y: p * y % MOD, vals, 1)

def max_product_cap_brute(nums, k, cap):
    best = -1
    for cs in product(range(min(cap, k) + 1), repeat=len(nums)):
        if sum(cs) <= k: best = max(best, prod(x + c for x, c in zip(nums, cs)))
    return best % MOD

rng6 = random.Random(11)
for _ in range(300):
    nums = [rng6.randint(0, 5) for _ in range(rng6.randint(1, 3))]; k = rng6.randint(1, 6); cap = rng6.randint(1, 4)
    assert max_product_cap_ans(nums, k, cap) == max_product_cap_brute(nums, k, cap), (nums, k, cap)
print("Bài 5: OK; [0, 9], k=5, cap=2 ->", max_product_cap_ans([0, 9], 5, 2))

Bài 5: OK; [0, 9], k=5, cap=2 -> 22


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
heapq.heapify(nums)
for _ in range(k):
    heapq.heapreplace(nums, nums[0] + 1)   # tăng min
ans = 1
for x in nums:
    ans = ans * x % MOD
```

In [37]:
def template(nums, k):
    h = list(nums); heapq.heapify(h)
    for _ in range(k):
        heapq.heapreplace(h, h[0] + 1)
    ans = 1
    for x in h:
        ans = ans * x % MOD
    return ans
print([template(n, k) for n, k, _ in examples])

[20, 216]


**Checklist:**

- [ ] Tăng phần tử **nhỏ nhất** (không phải lớn nhất).
- [ ] Dùng `heapreplace` (hoặc pop + push) — không sửa `h[0]` trực tiếp.
- [ ] Mod chỉ khi tính tích; không so sánh tích đã mod.
- [ ] Tích lấy mod sau mỗi phép nhân (tránh số khổng lồ).
- [ ] Nếu `k` cực lớn: đổ nước + `pow(x, e, MOD)`, nhớ phân phối phần dư `r`.
- [ ] Biên: mảng toàn 0, một phần tử, `k` nhỏ hơn số lượng số 0.